In [1]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_5497/398303453.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [4]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [5]:
!pip install sentence-transformers

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [11]:
vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    persist_directory='chroma-df_yo',
    collection_name='sample_hf'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
vector_store.add_documents(docs)
print("documents added to Chroma with HuggingFaceEmbeddings.")

documents added to Chroma with HuggingFaceEmbeddings.


In [15]:
#searching documents
vector_store.similarity_search(
    query="who among thhese are a bowler",
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [16]:
#search with similarity score
vector_store.similarity_search_with_score(
    query="who among thhese are a bowler",
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9437391757965088),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9437391757965088)]

In [17]:
#the lesser the value, more good cause this value represents distance, less distance, more close, more similar

In [18]:
#meta data filtering
vector_store.similarity_search(
    query="",
    filter={"team": "Mumbai Indians"}
)
#

[Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
 Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [19]:

# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [20]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['61b4c173-0be4-4e0d-9b81-bb6b6727d6fe',
  '7cdfb2d9-dbaa-4b65-8208-7c56128314a7',
  'fee20437-d68c-42c8-b7e4-b260bb9de835',
  '0d821f00-2d46-4942-b4c9-473e8d2b65f4',
  'a435683b-697a-4b2b-97da-8b3b5e3d829a',
  '0e713402-85cf-4b11-9c98-111d09bcd9a6',
  '160f077b-5810-49c7-8547-dc80e76265ca',
  '2a76c100-59ff-4698-80ac-1a65305b4869',
  '875ebc57-edab-4774-b06e-b430cd440e69',
  '604f2ebb-44bc-4786-835d-81cc3ebc1adf'],
 'embeddings': array([[ 0.00994725,  0.06914335, -0.0514712 , ..., -0.0354334 ,
          0.01284813,  0.01248285],
        [ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        ...,
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, 

In [21]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [22]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['61b4c173-0be4-4e0d-9b81-bb6b6727d6fe',
  '7cdfb2d9-dbaa-4b65-8208-7c56128314a7',
  'fee20437-d68c-42c8-b7e4-b260bb9de835',
  '0d821f00-2d46-4942-b4c9-473e8d2b65f4',
  'a435683b-697a-4b2b-97da-8b3b5e3d829a',
  '0e713402-85cf-4b11-9c98-111d09bcd9a6',
  '160f077b-5810-49c7-8547-dc80e76265ca',
  '2a76c100-59ff-4698-80ac-1a65305b4869',
  '875ebc57-edab-4774-b06e-b430cd440e69',
  '604f2ebb-44bc-4786-835d-81cc3ebc1adf'],
 'embeddings': array([[ 0.00994725,  0.06914335, -0.0514712 , ..., -0.0354334 ,
          0.01284813,  0.01248285],
        [ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        ...,
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, 